# Training Data 1 — Exploratory Data Analysis (EDA)

## Project Overview
This notebook explores the order, sales, profit, product, promotion, channel, and geographic fields in `TrainingData1.xlsx`.

**Workflow:** load data → inspect structure → assess data quality → create date-based fields → analyze KPIs and categories → explore trends and relationships → summarize findings.

> Keep `TrainingData1.xlsx` in the same folder as this notebook. In Google Colab, upload the Excel file when prompted.

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme()
plt.rcParams["figure.figsize"] = (10, 5)

file_path = Path("TrainingData1.xlsx")

# If running in Google Colab, upload the Excel file if it is not already present.
if not file_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next((name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))), None)
        if uploaded_name:
            file_path = Path(uploaded_name)
    except ImportError:
        pass

if not file_path.exists():
    raise FileNotFoundError(
        "TrainingData1.xlsx was not found. Place it beside this notebook or upload it in Google Colab."
    )

df = pd.read_excel(file_path, sheet_name="Data")
df.head()

## 2. Basic Structure

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nRandom sample:")
display(df.sample(min(5, len(df)), random_state=42))

In [ ]:
print("Numeric summary:")
display(df.describe().T)

print("Categorical summary:")
display(df.describe(include="object").T)

## 3. Data Quality Checks

In [ ]:
# Missing values
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percent": (df.isna().mean() * 100).round(2)
}).sort_values("Missing Count", ascending=False)
display(missing_summary)

# Duplicate rows and duplicate Order IDs
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Order IDs:", df["Order ID"].duplicated().sum())

# Validate data types
print("Order Date dtype before conversion:", df["Order Date"].dtype)
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
print("Invalid/missing dates after conversion:", df["Order Date"].isna().sum())

# Basic checks for negative quantities and monetary fields
for col in ["Unit Cost", "Price", "Order Qty", "Cost of Sales", "Sales", "Profit"]:
    if col in df.columns:
        print(f"Negative values in {col}: {(df[col] < 0).sum()}")

## 4. Derived Fields

In [ ]:
# Date-derived fields support time-based analysis.
df["Year"] = df["Order Date"].dt.year
df["Quarter"] = df["Order Date"].dt.to_period("Q").astype("string")
df["Month Number"] = df["Order Date"].dt.month
df["Month"] = df["Order Date"].dt.month_name()
df["Year-Month"] = df["Order Date"].dt.to_period("M").astype("string")
df["Day of Week"] = df["Order Date"].dt.day_name()
df["Day of Month"] = df["Order Date"].dt.day
df["Is Weekend"] = df["Order Date"].dt.dayofweek >= 5

display(df[["Order Date", "Year", "Quarter", "Month", "Year-Month",
            "Day of Week", "Day of Month", "Is Weekend"]].head())

## 5. Categorical Breakdowns

In [ ]:
categorical_cols = [
    "Channel", "Promotion Name", "Manufacturer", "Product Sub Category",
    "Product Category", "Region", "Country"
]

for col in categorical_cols:
    print(f"\\n{col}: {df[col].nunique()} unique values")
    display(df[col].value_counts(dropna=False).head(15).rename("Record Count").to_frame())

In [ ]:
# Order records by product category
category_counts = df["Product Category"].value_counts()
category_counts.plot(kind="bar")
plt.title("Order Records by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. Overall Business KPIs

In [ ]:
kpi_summary = pd.Series({
    "Order Records": len(df),
    "Unique Order IDs": df["Order ID"].nunique(),
    "Total Quantity": df["Order Qty"].sum(),
    "Total Sales": df["Sales"].sum(),
    "Total Cost of Sales": df["Cost of Sales"].sum(),
    "Total Profit": df["Profit"].sum(),
    "Average Sales per Record": df["Sales"].mean(),
    "Average Profit per Record": df["Profit"].mean(),
    "Overall Profit Margin (%)": (df["Profit"].sum() / df["Sales"].sum() * 100)
        if df["Sales"].sum() != 0 else np.nan
})
display(kpi_summary.to_frame("Value").round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
kpis = [
    ("Sales", df["Sales"].sum()),
    ("Cost of Sales", df["Cost of Sales"].sum()),
    ("Profit", df["Profit"].sum())
]
axes[0].bar([kpis[0][0]], [kpis[0][1]])
axes[1].bar([kpis[1][0]], [kpis[1][1]])
axes[2].bar([kpis[2][0]], [kpis[2][1]])
for ax, (label, value) in zip(axes, kpis):
    ax.set_title(f"Total {label}: {value:,.0f}")
    ax.set_ylabel("Amount")
plt.tight_layout()
plt.show()

## 7. Sales & Profit Over Time

In [ ]:
monthly = (
    df.dropna(subset=["Order Date"])
      .groupby("Year-Month", as_index=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Cost_of_Sales=("Cost of Sales", "sum"),
           Order_Records=("Order ID", "size"))
      .sort_values("Year-Month")
)

display(monthly.head())
display(monthly.tail())

plt.figure(figsize=(14, 5))
plt.plot(monthly["Year-Month"], monthly["Sales"], marker="o", label="Sales")
plt.plot(monthly["Year-Month"], monthly["Profit"], marker="o", label="Profit")
plt.title("Monthly Sales and Profit")
plt.xlabel("Year-Month")
plt.ylabel("Amount")
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
yearly = (
    df.groupby("Year", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Cost_of_Sales=("Cost of Sales", "sum"),
           Quantity=("Order Qty", "sum"),
           Order_Records=("Order ID", "size"))
      .reset_index()
)
display(yearly)

yearly.set_index("Year")[["Sales", "Profit"]].plot(kind="bar")
plt.title("Yearly Sales and Profit")
plt.xlabel("Year")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Product Category & Subcategory Analysis

In [ ]:
category_performance = (
    df.groupby("Product Category", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Cost_of_Sales=("Cost of Sales", "sum"),
           Quantity=("Order Qty", "sum"),
           Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
category_performance["Profit Margin (%)"] = (
    category_performance["Profit"] / category_performance["Sales"] * 100
).replace([np.inf, -np.inf], np.nan)
display(category_performance.round(2))

category_performance["Sales"].sort_values().plot(kind="barh")
plt.title("Sales by Product Category")
plt.xlabel("Sales")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

In [ ]:
subcategory_performance = (
    df.groupby("Product Sub Category", dropna=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
           Quantity=("Order Qty", "sum"), Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
display(subcategory_performance.head(15).round(2))

subcategory_performance.head(15).sort_values("Sales").plot(kind="barh", y="Sales", legend=False)
plt.title("Top 15 Subcategories by Sales")
plt.xlabel("Sales")
plt.ylabel("Product Sub Category")
plt.tight_layout()
plt.show()

## 9. Channel & Promotion Analysis

In [ ]:
channel_performance = (
    df.groupby("Channel", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Cost_of_Sales=("Cost of Sales", "sum"),
           Quantity=("Order Qty", "sum"),
           Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
channel_performance["Profit Margin (%)"] = (
    channel_performance["Profit"] / channel_performance["Sales"] * 100
).replace([np.inf, -np.inf], np.nan)
display(channel_performance.round(2))

channel_performance[["Sales", "Profit"]].plot(kind="bar")
plt.title("Sales and Profit by Channel")
plt.xlabel("Channel")
plt.ylabel("Amount")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
promotion_performance = (
    df.groupby("Promotion Name", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Quantity=("Order Qty", "sum"),
           Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
promotion_performance["Profit Margin (%)"] = (
    promotion_performance["Profit"] / promotion_performance["Sales"] * 100
).replace([np.inf, -np.inf], np.nan)
display(promotion_performance.round(2))

promotion_performance["Sales"].sort_values().plot(kind="barh")
plt.title("Sales by Promotion")
plt.xlabel("Sales")
plt.ylabel("Promotion Name")
plt.tight_layout()
plt.show()

## 10. Geographic Analysis

In [ ]:
region_performance = (
    df.groupby("Region", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Quantity=("Order Qty", "sum"),
           Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
region_performance["Profit Margin (%)"] = (
    region_performance["Profit"] / region_performance["Sales"] * 100
).replace([np.inf, -np.inf], np.nan)
display(region_performance.round(2))

region_performance[["Sales", "Profit"]].plot(kind="bar")
plt.title("Sales and Profit by Region")
plt.xlabel("Region")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
country_performance = (
    df.groupby("Country", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Quantity=("Order Qty", "sum"),
           Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
display(country_performance.head(15).round(2))

country_performance.head(15).sort_values("Sales").plot(kind="barh", y="Sales", legend=False)
plt.title("Top 15 Countries by Sales")
plt.xlabel("Sales")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 11. Manufacturer, Relationships & Validation

In [ ]:
manufacturer_performance = (
    df.groupby("Manufacturer", dropna=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Quantity=("Order Qty", "sum"),
           Records=("Order ID", "size"))
      .sort_values("Sales", ascending=False)
)
manufacturer_performance["Profit Margin (%)"] = (
    manufacturer_performance["Profit"] / manufacturer_performance["Sales"] * 100
).replace([np.inf, -np.inf], np.nan)
display(manufacturer_performance.round(2))

manufacturer_performance[["Sales", "Profit"]].plot(kind="bar")
plt.title("Sales and Profit by Manufacturer")
plt.xlabel("Manufacturer")
plt.ylabel("Amount")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between numeric fields
numeric_cols = ["Unit Cost", "Price", "Order Qty", "Cost of Sales", "Sales", "Profit"]
corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numeric Fields")
plt.tight_layout()
plt.show()

In [ ]:
# Check whether the recorded Sales/Cost of Sales values match simple price/quantity calculations.
# These are diagnostic checks only; the original fields are not overwritten.
df["Calculated Sales (Price x Qty)"] = df["Price"] * df["Order Qty"]
df["Sales Difference"] = df["Sales"] - df["Calculated Sales (Price x Qty)"]
df["Calculated Cost (Unit Cost x Qty)"] = df["Unit Cost"] * df["Order Qty"]
df["Cost Difference"] = df["Cost of Sales"] - df["Calculated Cost (Unit Cost x Qty)"]

validation_summary = pd.DataFrame({
    "Metric": ["Sales difference is approximately zero",
               "Cost difference is approximately zero"],
    "Matching Rows": [
        np.isclose(df["Sales Difference"], 0, atol=0.01).sum(),
        np.isclose(df["Cost Difference"], 0, atol=0.01).sum()
    ],
    "Total Rows": [len(df), len(df)]
})
validation_summary["Matching Percent"] = (
    validation_summary["Matching Rows"] / validation_summary["Total Rows"] * 100
).round(2)
display(validation_summary)

# Profit distribution and potential outliers
plt.figure(figsize=(10, 4))
sns.boxplot(x=df["Profit"])
plt.title("Profit Distribution and Potential Outliers")
plt.xlabel("Profit")
plt.tight_layout()
plt.show()

display(df[["Sales", "Cost of Sales", "Profit"]].describe().T.round(2))

## 12. Summary & Key Findings

In [ ]:
print("DATASET SUMMARY")
print("-" * 50)
print(f"Rows: {len(df):,}")
print(f"Columns (including derived/validation fields): {df.shape[1]}")
print(f"Order date range: {df['Order Date'].min()} to {df['Order Date'].max()}")
print(f"Unique Order IDs: {df['Order ID'].nunique():,}")
print(f"Total quantity: {df['Order Qty'].sum():,.0f}")
print(f"Total sales: {df['Sales'].sum():,.2f}")
print(f"Total cost of sales: {df['Cost of Sales'].sum():,.2f}")
print(f"Total profit: {df['Profit'].sum():,.2f}")
profit_margin = df['Profit'].sum() / df['Sales'].sum() * 100 if df['Sales'].sum() else np.nan
print(f"Overall profit margin: {profit_margin:.2f}%")
print(f"Exact duplicate rows in original data: {df.duplicated(subset=['Order ID', 'Order Date', 'Unit Cost', 'Price', 'Order Qty', 'Cost of Sales', 'Sales', 'Profit', 'Channel', 'Promotion Name', 'Product Name', 'Manufacturer', 'Product Sub Category', 'Product Category', 'Region', 'City', 'Country']).sum()}")

print("\\nTop categories by sales:")
display(category_performance.head(5).round(2))
print("\\nTop channels by sales:")
display(channel_performance.head(5).round(2))
print("\\nTop regions by sales:")
display(region_performance.head(5).round(2))

print("\\nInterpretation note: Use the tables and charts above to identify strong/weak categories, profitable channels, geographic patterns, and whether sales/cost calculations align with the source data.")